In [5]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.metrics.pairwise import cosine_similarity

In [6]:
# Đọc dữ liệu
df = pd.read_csv('./Products.csv')
df.head(3)

,ProductKey,Product Name,Brand,Color,Unit Cost USD,Unit Price USD,SubcategoryKey,Subcategory,CategoryKey,Category
0,1,Contoso 512MB MP3 Player E51 Silver,Contoso,Silver,$6.62,$12.99,101,MP4&MP3,1,Audio
1,2,Contoso 512MB MP3 Player E51 Blue,Contoso,Blue,$6.62,$12.99,101,MP4&MP3,1,Audio
2,3,Contoso 1G MP3 Player E100 White,Contoso,White,$7.40,$14.52,101,MP4&MP3,1,Audio


In [7]:
main_feature4m_df = df[['ProductKey','Product Name', 'Brand', 'Color','Subcategory', 'Category']].copy()
main_feature4m_df['product_details'] = (
    main_feature4m_df['Product Name'] + ' ' + 
    main_feature4m_df['Brand'] + ' ' + 
    main_feature4m_df['Color'] + ' ' + 
    main_feature4m_df['Subcategory'] + ' ' + 
    main_feature4m_df['Category'] 
)

In [8]:
tfidf_vec = TfidfVectorizer(stop_words='english', max_df=0.95, min_df=2, ngram_range=(1, 1))
tfidf_matrix = tfidf_vec.fit_transform(main_feature4m_df['product_details'])

In [9]:
n_topics = 10  
lda = LatentDirichletAllocation(n_components=n_topics, random_state=42)
lda_matrix = lda.fit_transform(tfidf_matrix)

In [10]:
similarity_matrix = cosine_similarity(lda_matrix)

In [11]:
def plsa_based_recommendations(product_index, similarity_matrix, data, top_n=10):
    # Sắp xếp các sản phẩm theo mức độ tương đồng giảm dần
    similar_indices = similarity_matrix[product_index].argsort()[::-1]
    # Loại bỏ chính sản phẩm được chọn
    similar_indices = similar_indices[similar_indices != product_index]
    # Lấy top_n sản phẩm tương tự
    top_indices = similar_indices[:top_n]
    return data.iloc[top_indices], similarity_matrix[product_index][top_indices]


In [13]:
try:
    input_product_index = int(input("Product Index: "))
    if input_product_index < 0 or input_product_index >= len(main_feature4m_df):
        raise ValueError("Chỉ số sai.")
except ValueError as e:
    print(f"Lỗi: {e}")
    exit()

recommendations_result, similarities = plsa_based_recommendations(input_product_index, similarity_matrix, main_feature4m_df, top_n=10)

# Hiển thị kết quả
print("\nĐề xuất:")
print(recommendations_result[['product_details']])


Đề xuất:
                                        product_details
38    Contoso 8GB Clock & Radio MP3 Player X850 Gree...
39    Contoso 8GB Clock & Radio MP3 Player X850 Blue...
1     Contoso 512MB MP3 Player E51 Blue Contoso Blue...
36    Contoso 8GB Clock & Radio MP3 Player X850 Silv...
37    Contoso 8GB Clock & Radio MP3 Player X850 Blac...
1299  Contoso Telephoto Conversion Lens M350 Black C...
2     Contoso 1G MP3 Player E100 White Contoso White...
10    Contoso 4G MP3 Player E400 Orange Contoso Oran...
4     Contoso 2G MP3 Player E200 Red Contoso Red MP4...
9     Contoso 4G MP3 Player E400 Green Contoso Green...
